In [ ]:
from enum import Enum

from pydantic import BaseModel


class RhythmType(str, Enum):
    REGULAR = "regular"
    IRREGULAR = "irregular"
    
class LanguageUnderstanding(str, Enum):
    YES = "english_competence"
    NO = "non_english_competence"

class Sample(BaseModel):
    participant: int
    line: int
    set_path: str
    fdt_path: str
    rhythm_type: RhythmType
    language_understanding: LanguageUnderstanding
    


In [ ]:
from adapt_eeg.constants import ENGLISH_COMPETENCE_PARTICIPANTS, REGULAR_LINES,IRREGULAR_LINES

def rhythm_type(line_number: int) -> RhythmType:
    if line_number in REGULAR_LINES:
        return RhythmType.REGULAR
    if line_number in IRREGULAR_LINES:
        return RhythmType.IRREGULAR
    raise ValueError(f"Line {line_number} is not a constants value of either regular or irregular type")


def language_understanding(participant_id: int) -> LanguageUnderstanding:
    if participant_id in ENGLISH_COMPETENCE_PARTICIPANTS:
        return LanguageUnderstanding.YES
    return LanguageUnderstanding.NO

In [ ]:
import logging
from pathlib import Path
import re

FILE_RE = re.compile(r"part(?P<participant>\d+)v(?P<line>\d+)\.set$")


data_root = Path("../data/raw/itpc-30event-good")

samples: list[Sample] = []
for set_path in data_root.glob("line*/*.set"):
    match = FILE_RE.search(set_path.name)
    if not match:
        logging.info("Skipping file %s. Does not match format", set_path.name)
        continue

    participant = int(match.group("participant"))
    line = int(match.group("line"))
    fdt_path = set_path.with_suffix(".fdt")

    samples.append(
        Sample(
            participant=participant,
            line=line,
            set_path=str(set_path),
            fdt_path=str(fdt_path),
            rhythm_type=rhythm_type(line),
            language_understanding=language_understanding(participant),
        )
    )

In [ ]:
import mne


def process(sample):
    eeg = mne.read_epochs_eeglab(
        sample.set_path,
        verbose="warning",
    )
    return eeg


sample = samples[113]
epochs = process(sample)
epochs

In [ ]:
def describe_epochs(sample: Sample, epochs: mne.Epochs) -> None:
    data_shape = epochs.get_data(copy=False).shape

    print("=" * 80)
    print(f"Participant: {sample.participant}")
    print(f"Line: {sample.line}")
    print(f"Rhythm: {sample.rhythm_type.value}")
    print(f"Language group: {sample.language_understanding.value}")
    print(f"Object type: {type(epochs).__name__}")
    print(f"Number of trials: {len(epochs)}")
    print(f"Number of channels: {len(epochs.ch_names)}")
    print(f"Sampling frequency: {epochs.info['sfreq']} Hz")
    print(f"Epoch start: {epochs.tmin:.3f} s")
    print(f"Epoch end: {epochs.tmax:.3f} s")
    print(f"Epoch events: {epochs.events}")
    print(f"Samples per epoch: {len(epochs.times)}")
    print(f"Data shape: {data_shape}")
    print(f"Bad channels: {epochs.info['bads']}")
    print(f"Event IDs: {epochs.event_id}")
    print(f"First five events:\n{epochs.events[:5]}")
    print(f"Channel names:\n{epochs.ch_names}")

describe_epochs(sample, epochs)

In [ ]:
epochs.get_annotations_per_epoch

In [ ]:
epochs.plot(
    n_epochs=5,
    n_channels=5,
    scalings="auto",
)

In [ ]:
regular_sample = next(
    sample for sample in samples if sample.rhythm_type == RhythmType.REGULAR
)

irregular_sample = next(
    sample for sample in samples if sample.rhythm_type == RhythmType.IRREGULAR
)

In [ ]:
from scipy.io import loadmat

set_info = loadmat(
    regular_sample.set_path,
    simplify_cells=True,
)

print(set_info.keys())

eeg = set_info.get("EEG", set_info)

print("Available fields:")
print(eeg.keys())

In [ ]:
fields = [
    "setname",
    "filename",
    "filepath",
    "subject",
    "condition",
    "comments",
    "nbchan",
    "trials",
    "pnts",
    "srate",
    "xmin",
    "xmax",
    "times",
    "event",
    "epoch",
    "urevent",
    "history",
]

for field in fields:
    value = eeg.get(field, "<missing>")
    print(f"\n--- {field} ---")
    print(value)

In [ ]:
from __future__ import annotations

import numpy as np
import mne

# Initial estimate from the imported EEGLAB event timing.
DEFAULT_EPOCH_SHIFT_SECONDS = 0.0843109


def reconstruct_original_sequence(
    epochs: mne.Epochs,
) -> mne.io.RawArray:
    """
    Reconstruct an approximate continuous sequence from overlapping epochs.

    The epoch-to-epoch shift is automatically refined around
    DEFAULT_EPOCH_SHIFT_SECONDS by finding the shift that minimizes the
    mismatch between overlapping epochs.

    Parameters
    ----------
    epochs
        MNE Epochs object containing sequential overlapping windows.

    Returns
    -------
    raw
        Reconstructed continuous MNE RawArray.

    Notes
    -----
    Each epoch was independently baseline-corrected. Therefore, the original
    absolute DC level cannot be recovered exactly. This function estimates and
    removes a constant channel-wise offset between overlapping epochs.
    """
    data = epochs.get_data(copy=True)

    if data.ndim != 3:
        raise ValueError("Expected data shaped as " "(n_epochs, n_channels, n_times).")

    n_epochs, n_channels, n_times = data.shape
    sfreq = float(epochs.info["sfreq"])
    epoch_times = np.asarray(epochs.times, dtype=float)

    if n_epochs < 2:
        raise ValueError("At least two epochs are required.")

    epoch_duration = epoch_times[-1] - epoch_times[0]

    # Refine the approximate event displacement.
    shift_seconds, shift_error = _find_best_shift(
        data=data,
        epoch_times=epoch_times,
        sfreq=sfreq,
        initial_shift=DEFAULT_EPOCH_SHIFT_SECONDS,
    )

    if shift_seconds >= epoch_duration:
        raise ValueError(
            "The estimated shift is greater than or equal to the epoch "
            "duration, so consecutive epochs do not overlap."
        )

    print(
        f"Initial shift: {DEFAULT_EPOCH_SHIFT_SECONDS:.9f} s "
        f"({DEFAULT_EPOCH_SHIFT_SECONDS * sfreq:.4f} samples)"
    )
    print(
        f"Optimized shift: {shift_seconds:.9f} s "
        f"({shift_seconds * sfreq:.4f} samples)"
    )
    print(f"Overlap-fit error: {shift_error:.6e}")

    epoch_centres = np.arange(n_epochs, dtype=float) * shift_seconds

    absolute_times = epoch_centres[:, None] + epoch_times[None, :]

    # Correct relative offsets caused by independent baseline correction.
    aligned = data.copy()

    for epoch_index in range(1, n_epochs):
        offset = _estimate_epoch_offset(
            reference_epoch=aligned[epoch_index - 1],
            current_epoch=aligned[epoch_index],
            reference_times=absolute_times[epoch_index - 1],
            current_times=absolute_times[epoch_index],
        )

        aligned[epoch_index] += offset[:, None]

    # Construct a common output timeline.
    output_start = absolute_times[0, 0]
    output_end = absolute_times[-1, -1]

    n_output_samples = int(np.round((output_end - output_start) * sfreq)) + 1

    output_times = output_start + np.arange(n_output_samples, dtype=float) / sfreq

    accumulated = np.zeros(
        (n_channels, n_output_samples),
        dtype=float,
    )

    weights = np.zeros(
        n_output_samples,
        dtype=float,
    )

    # Interpolate and average overlapping windows.
    for epoch_index in range(n_epochs):
        source_times = absolute_times[epoch_index]

        valid_mask = (output_times >= source_times[0]) & (
            output_times <= source_times[-1]
        )

        target_times = output_times[valid_mask]

        for channel_index in range(n_channels):
            accumulated[channel_index, valid_mask] += np.interp(
                target_times,
                source_times,
                aligned[epoch_index, channel_index],
            )

        weights[valid_mask] += 1.0

    if np.any(weights == 0):
        missing = int(np.sum(weights == 0))
        raise RuntimeError(
            f"The reconstructed sequence contains {missing} empty samples."
        )

    reconstructed = accumulated / weights[None, :]

    info = epochs.info.copy()

    raw = mne.io.RawArray(
        reconstructed,
        info,
        first_samp=0,
        verbose=False,
    )

    raw.info["description"] = (
        "Approximate continuous EEG reconstructed from sequential "
        "overlapping epochs. "
        f"Optimized epoch shift={shift_seconds:.9f} seconds; "
        f"shift={shift_seconds * sfreq:.4f} samples; "
        f"overlap error={shift_error:.6e}; "
        f"original relative start={output_start:.9f} seconds."
    )

    return raw


def _find_best_shift(
    data: np.ndarray,
    epoch_times: np.ndarray,
    sfreq: float,
    initial_shift: float,
) -> tuple[float, float]:
    """
    Refine the epoch shift by minimizing overlap mismatch.

    A two-stage grid search is used:

    1. Coarse search within ±2 samples of the initial estimate.
    2. Fine search around the best coarse result.

    Returns
    -------
    best_shift
        Estimated shift in seconds.

    best_error
        Median normalized overlap error.
    """
    sample_period = 1.0 / sfreq

    # Search within ±2 original EEG samples.
    coarse_candidates = np.linspace(
        initial_shift - 2.0 * sample_period,
        initial_shift + 2.0 * sample_period,
        161,
    )

    coarse_errors = np.array(
        [
            _shift_error(
                data=data,
                epoch_times=epoch_times,
                shift_seconds=shift,
            )
            for shift in coarse_candidates
        ]
    )

    valid = np.isfinite(coarse_errors)

    if not np.any(valid):
        raise RuntimeError("No valid overlapping shift could be estimated.")

    coarse_best_index = np.nanargmin(coarse_errors)
    coarse_best_shift = coarse_candidates[coarse_best_index]

    coarse_step = coarse_candidates[1] - coarse_candidates[0]

    # Fine search around the best coarse result.
    fine_candidates = np.linspace(
        coarse_best_shift - coarse_step,
        coarse_best_shift + coarse_step,
        201,
    )

    fine_errors = np.array(
        [
            _shift_error(
                data=data,
                epoch_times=epoch_times,
                shift_seconds=shift,
            )
            for shift in fine_candidates
        ]
    )

    best_index = np.nanargmin(fine_errors)

    return (
        float(fine_candidates[best_index]),
        float(fine_errors[best_index]),
    )


def _shift_error(
    data: np.ndarray,
    epoch_times: np.ndarray,
    shift_seconds: float,
) -> float:
    """
    Calculate robust mismatch between consecutive overlapping epochs.

    Constant channel-wise offsets are removed before calculating the error,
    because each epoch was independently baseline-corrected.
    """
    if shift_seconds <= 0:
        return np.inf

    epoch_duration = epoch_times[-1] - epoch_times[0]

    if shift_seconds >= epoch_duration:
        return np.inf

    n_epochs = data.shape[0]
    pair_errors: list[float] = []

    for epoch_index in range(1, n_epochs):
        reference_times = epoch_times
        current_times = epoch_times + shift_seconds

        overlap_start = max(
            reference_times[0],
            current_times[0],
        )
        overlap_end = min(
            reference_times[-1],
            current_times[-1],
        )

        if overlap_end <= overlap_start:
            continue

        current_mask = (current_times >= overlap_start) & (current_times <= overlap_end)

        comparison_times = current_times[current_mask]

        if comparison_times.size < 5:
            continue

        reference_overlap = np.vstack(
            [
                np.interp(
                    comparison_times,
                    reference_times,
                    data[epoch_index - 1, channel_index],
                )
                for channel_index in range(data.shape[1])
            ]
        )

        current_overlap = data[epoch_index][:, current_mask]

        difference = reference_overlap - current_overlap

        # Remove the unknown constant introduced by baseline correction.
        difference -= np.median(
            difference,
            axis=1,
            keepdims=True,
        )

        # Normalize by robust signal scale so high-amplitude channels do not
        # dominate the shift estimation.
        combined = np.concatenate(
            [reference_overlap, current_overlap],
            axis=1,
        )

        channel_scale = np.median(
            np.abs(combined - np.median(combined, axis=1, keepdims=True)),
            axis=1,
        )

        channel_scale = np.maximum(
            channel_scale,
            np.finfo(float).eps,
        )

        normalized_error = np.median(np.abs(difference), axis=1) / channel_scale

        pair_errors.append(float(np.median(normalized_error)))

    if not pair_errors:
        return np.inf

    return float(np.median(pair_errors))


def _estimate_epoch_offset(
    reference_epoch: np.ndarray,
    current_epoch: np.ndarray,
    reference_times: np.ndarray,
    current_times: np.ndarray,
) -> np.ndarray:
    """
    Estimate the constant channel-wise offset between two overlapping epochs.
    """
    overlap_start = max(
        reference_times[0],
        current_times[0],
    )
    overlap_end = min(
        reference_times[-1],
        current_times[-1],
    )

    if overlap_end <= overlap_start:
        raise ValueError("Consecutive epochs do not overlap.")

    current_mask = (current_times >= overlap_start) & (current_times <= overlap_end)

    comparison_times = current_times[current_mask]

    reference_overlap = np.vstack(
        [
            np.interp(
                comparison_times,
                reference_times,
                reference_epoch[channel_index],
            )
            for channel_index in range(reference_epoch.shape[0])
        ]
    )

    current_overlap = current_epoch[:, current_mask]

    return np.median(
        reference_overlap - current_overlap,
        axis=1,
    )

In [ ]:
reconstructed = reconstruct_original_sequence(epochs)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_epoch_overlap(
    epochs,
    epoch_index: int,
    shift_samples: int,
    channel: int | str = 0,
    correct_offset: bool = True,
) -> None:
    """
    Overlay the tail of one epoch with the beginning of the next epoch.

    Parameters
    ----------
    epochs
        MNE Epochs object.

    epoch_index
        Index of the first epoch. The function compares:
        epoch_index and epoch_index + 1.

    shift_samples
        Number of samples by which the next epoch starts later.
        Usually 21 or 22 for this dataset.

    channel
        Channel index or channel name.

    correct_offset
        Whether to remove the constant baseline difference between epochs.
    """
    data = epochs.get_data(copy=True)
    sfreq = float(epochs.info["sfreq"])

    if epoch_index < 0 or epoch_index >= len(epochs) - 1:
        raise IndexError("epoch_index must identify an epoch that has a next epoch.")

    if isinstance(channel, str):
        if channel not in epochs.ch_names:
            raise ValueError(f"Unknown channel: {channel}")
        channel_index = epochs.ch_names.index(channel)
        channel_name = channel
    else:
        channel_index = int(channel)
        channel_name = epochs.ch_names[channel_index]

    n_times = data.shape[-1]

    if shift_samples <= 0 or shift_samples >= n_times:
        raise ValueError(f"shift_samples must be between 1 and {n_times - 1}.")

    first = data[epoch_index, channel_index]
    second = data[epoch_index + 1, channel_index]

    # The shared physical signal is:
    # first[shift_samples:] versus second[:n_times-shift_samples]
    first_overlap = first[shift_samples:].copy()
    second_overlap = second[: n_times - shift_samples].copy()

    offset = 0.0

    if correct_offset:
        offset = np.median(first_overlap - second_overlap)
        second_overlap += offset

    overlap_times_ms = np.arange(len(first_overlap)) / sfreq * 1000

    correlation = np.corrcoef(
        first_overlap,
        second_overlap,
    )[0, 1]

    rmse = np.sqrt(np.mean((first_overlap - second_overlap) ** 2))

    plt.figure(figsize=(12, 5))

    plt.plot(
        overlap_times_ms,
        first_overlap,
        label=f"Epoch {epoch_index}, shifted tail",
        linewidth=2,
    )

    plt.plot(
        overlap_times_ms,
        second_overlap,
        label=f"Epoch {epoch_index + 1}, beginning",
        linewidth=2,
        alpha=0.8,
    )

    plt.xlabel("Time within overlap (ms)")
    plt.ylabel("EEG amplitude (V)")
    plt.title(
        f"{channel_name}: epochs {epoch_index} and {epoch_index + 1}\n"
        f"shift={shift_samples} samples "
        f"({shift_samples / sfreq * 1000:.2f} ms), "
        f"correlation={correlation:.3f}, "
        f"RMSE={rmse:.3e}"
    )

    plt.legend()
    plt.tight_layout()
    plt.show()

    print(f"Channel: {channel_name}")
    print(f"Shift: {shift_samples} samples")
    print(f"Shift: {shift_samples / sfreq:.9f} seconds")
    print(f"Baseline offset applied: {offset:.6e} V")
    print(f"Overlap correlation: {correlation:.6f}")
    print(f"Overlap RMSE: {rmse:.6e} V")

In [ ]:
channels = epochs.ch_names

for c in channels:

    plot_epoch_overlap(
        epochs,
        epoch_index=4,
        shift_samples=19,
        channel=c,
    )
    break

In [ ]:
import numpy as np
import pandas as pd


def diagnose_epoch_alignment(
    epochs,
    shift_samples: int = 20,
) -> pd.DataFrame:
    """
    Score every adjacent epoch pair separately.

    Higher correlation is better.
    Lower normalized error is better.
    """
    data = epochs.get_data(copy=True)
    n_epochs, n_channels, n_times = data.shape

    if shift_samples <= 0 or shift_samples >= n_times:
        raise ValueError("Invalid shift_samples.")

    rows = []

    for i in range(n_epochs - 1):
        reference = data[i, :, shift_samples:]
        current = data[i + 1, :, : n_times - shift_samples]

        # Remove the constant offset introduced by independent baselines.
        offset = np.median(
            reference - current,
            axis=1,
            keepdims=True,
        )

        current_aligned = current + offset
        difference = reference - current_aligned

        channel_correlations = []
        channel_errors = []

        for ch in range(n_channels):
            x = reference[ch]
            y = current_aligned[ch]

            if np.std(x) > 0 and np.std(y) > 0:
                channel_correlations.append(np.corrcoef(x, y)[0, 1])

            scale = np.median(np.abs(x - np.median(x)))

            if scale > np.finfo(float).eps:
                channel_errors.append(np.median(np.abs(x - y)) / scale)

        rows.append(
            {
                "pair": f"{i}->{i + 1}",
                "epoch_before": i,
                "epoch_after": i + 1,
                "median_correlation": np.median(channel_correlations),
                "median_normalized_error": np.median(channel_errors),
                "max_channel_error": np.max(channel_errors),
            }
        )

    return pd.DataFrame(rows)

diagnostics = diagnose_epoch_alignment(
    epochs,
    shift_samples=19,
)

print(
    diagnostics.sort_values(
        "median_normalized_error",
        ascending=False,
    ).to_string(index=False)
)

In [55]:
from __future__ import annotations

import numpy as np
import mne


def merge_sliding_epochs(
    epochs: mne.Epochs,
    min_shift: int = 18,
    max_shift: int = 23,
    absolute_tolerance: float = 1e-12,
    normalized_tolerance: float = 1e-5,
    verbose: bool = False,
) -> tuple[mne.EpochsArray, dict]:
    """
    Merge sequential overlapping epochs into one reconstructed epoch.

    For each adjacent pair, integer shifts from `min_shift` through
    `max_shift` are tested. A shift is accepted only when the overlapping
    samples match up to floating-point numerical error.

    A constant offset is estimated separately for every channel because the
    original EEGLAB epochs were independently baseline-corrected.

    Parameters
    ----------
    epochs
        Sequential MNE epochs with shape:
        (n_epochs, n_channels, n_times).

    min_shift
        Smallest candidate shift in samples.

    max_shift
        Largest candidate shift in samples, inclusive.

    absolute_tolerance
        Absolute residual below which differences are treated as numerical
        noise.

    normalized_tolerance
        Maximum permitted residual relative to the robust signal amplitude.
        A value such as 1e-5 permits tiny floating-point discrepancies but
        rejects genuinely mismatched signals.

    verbose
        Print the selected shift and matching error for each pair.

    Returns
    -------
    merged_epochs
        An MNE EpochsArray containing one reconstructed epoch with shape:
        (1, n_channels, reconstructed_n_times).

    diagnostics
        Dictionary containing pairwise shifts, errors, offsets, and epoch
        start positions.

    Raises
    ------
    RuntimeError
        If any adjacent pair has no perfect matching shift in the requested
        range.
    """
    data = epochs.get_data(copy=True)

    if data.ndim != 3:
        raise ValueError("Expected data with shape " "(n_epochs, n_channels, n_times).")

    n_epochs, n_channels, n_times = data.shape
    sfreq = float(epochs.info["sfreq"])

    if n_epochs < 2:
        raise ValueError("At least two epochs are required.")

    if min_shift < 1:
        raise ValueError("min_shift must be at least 1.")

    if max_shift < min_shift:
        raise ValueError("max_shift must be >= min_shift.")

    if max_shift >= n_times:
        raise ValueError(
            f"max_shift must be less than the epoch length " f"({n_times} samples)."
        )

    aligned_epochs = data.copy()

    selected_shifts: list[int] = []
    selected_errors: list[float] = []
    selected_absolute_errors: list[float] = []
    selected_offsets: list[np.ndarray] = []

    for pair_index in range(n_epochs - 1):
        reference = aligned_epochs[pair_index]
        current_original = data[pair_index + 1]

        candidates: list[dict] = []

        for shift in range(min_shift, max_shift + 1):
            overlap_length = n_times - shift

            reference_overlap = reference[:, shift:]
            current_overlap = current_original[:, :overlap_length]

            # Correct the constant difference introduced by independent
            # baseline correction.
            offset = np.median(
                reference_overlap - current_overlap,
                axis=1,
            )

            current_corrected = current_overlap + offset[:, None]

            residual = reference_overlap - current_corrected

            max_absolute_error = float(np.max(np.abs(residual)))

            # Robust channel scale based on median absolute deviation.
            channel_scale = np.median(
                np.abs(
                    reference_overlap
                    - np.median(
                        reference_overlap,
                        axis=1,
                        keepdims=True,
                    )
                ),
                axis=1,
            )

            # Avoid division by zero for flat channels.
            channel_scale = np.maximum(
                channel_scale,
                absolute_tolerance,
            )

            channel_normalized_error = np.max(np.abs(residual), axis=1) / channel_scale

            normalized_error = float(np.median(channel_normalized_error))

            # Accept either truly tiny absolute residuals or tiny residuals
            # relative to the EEG amplitude.
            is_perfect_match = (
                max_absolute_error <= absolute_tolerance
                or normalized_error <= normalized_tolerance
            )

            candidates.append(
                {
                    "shift": shift,
                    "offset": offset,
                    "absolute_error": max_absolute_error,
                    "normalized_error": normalized_error,
                    "is_perfect": is_perfect_match,
                }
            )

        perfect_candidates = [
            candidate for candidate in candidates if candidate["is_perfect"]
        ]

        if not perfect_candidates:
            candidate_summary = "\n".join(
                (
                    f"  shift={candidate['shift']:2d}: "
                    f"absolute_error="
                    f"{candidate['absolute_error']:.6e}, "
                    f"normalized_error="
                    f"{candidate['normalized_error']:.6e}"
                )
                for candidate in candidates
            )

            raise RuntimeError(
                f"No perfect overlap found for epochs "
                f"{pair_index}->{pair_index + 1}.\n"
                f"Tested shifts {min_shift} through {max_shift}.\n"
                f"Candidate errors:\n{candidate_summary}"
            )

        # Among acceptable matches, select the smallest normalized residual.
        best = min(
            perfect_candidates,
            key=lambda candidate: (
                candidate["normalized_error"],
                candidate["absolute_error"],
            ),
        )

        shift = int(best["shift"])
        offset = np.asarray(best["offset"])

        # Apply the channel-wise offset to the complete next epoch.
        aligned_epochs[pair_index + 1] = current_original + offset[:, None]

        selected_shifts.append(shift)
        selected_offsets.append(offset)
        selected_errors.append(float(best["normalized_error"]))
        selected_absolute_errors.append(float(best["absolute_error"]))

        if verbose:
            print(
                f"{pair_index:02d}->{pair_index + 1:02d}: "
                f"shift={shift:2d} samples "
                f"({shift / sfreq * 1000:.3f} ms), "
                f"normalized_error="
                f"{best['normalized_error']:.3e}, "
                f"absolute_error="
                f"{best['absolute_error']:.3e}"
            )

    # Construct the cumulative start sample for every epoch.
    starts = np.zeros(n_epochs, dtype=int)
    starts[1:] = np.cumsum(selected_shifts)

    reconstructed_length = int(starts[-1] + n_times)

    accumulated = np.zeros(
        (n_channels, reconstructed_length),
        dtype=float,
    )

    weights = np.zeros(
        reconstructed_length,
        dtype=float,
    )

    for epoch_index, start in enumerate(starts):
        stop = start + n_times

        accumulated[:, start:stop] += aligned_epochs[epoch_index]
        weights[start:stop] += 1.0

    if np.any(weights == 0):
        missing_samples = np.flatnonzero(weights == 0)

        raise RuntimeError(
            "The reconstructed sequence contains gaps at samples "
            f"{missing_samples.tolist()}."
        )

    reconstructed = accumulated / weights[None, :]

    # Create one merged epoch. Preserve the original epoch start time.
    merged_data = reconstructed[None, :, :]

    merged_events = np.array(
        [[0, 0, 1]],
        dtype=int,
    )

    merged_epochs = mne.EpochsArray(
        merged_data,
        epochs.info.copy(),
        events=merged_events,
        event_id={"reconstructed_sequence": 1},
        tmin=float(epochs.tmin),
        baseline=None,
        verbose=False,
    )

    diagnostics = {
        "pairwise_shifts": np.asarray(
            selected_shifts,
            dtype=int,
        ),
        "epoch_starts": starts,
        "normalized_errors": np.asarray(
            selected_errors,
            dtype=float,
        ),
        "absolute_errors": np.asarray(
            selected_absolute_errors,
            dtype=float,
        ),
        "channel_offsets": np.asarray(
            selected_offsets,
            dtype=float,
        ),
        "weights": weights,
        "sampling_frequency": sfreq,
    }

    if verbose:
        shifts = diagnostics["pairwise_shifts"]

        print("\nReconstruction complete")
        print(f"Original epochs: {n_epochs}")
        print(f"Original epoch length: {n_times} samples")
        print(f"Reconstructed length: " f"{reconstructed_length} samples")
        print(f"Reconstructed duration: " f"{(reconstructed_length - 1) / sfreq:.6f} s")
        print(f"Shift sequence: {shifts.tolist()}")
        print(f"Mean shift: {shifts.mean():.4f} samples")
        print(
            "Maximum normalized error: " f"{diagnostics['normalized_errors'].max():.3e}"
        )

    return merged_epochs, diagnostics

In [56]:
merged_epoch, diagnostics = merge_sliding_epochs(
    epochs,
    min_shift=18,
    max_shift=25,
)

print(merged_epoch)
print(merged_epoch.get_data().shape)

<EpochsArray | 1 events (all good), -0.102 – 2.09 s (baseline off), ~104 KiB, data loaded,
 'reconstructed_sequence': 1>
(1, 18, 562)


In [57]:
diagnostics

{'pairwise_shifts': array([20, 19, 19, 20, 19, 19, 20, 19, 20, 19, 19, 20, 19, 19, 20, 19, 20,
        19, 19, 20, 19, 19, 20, 19, 20]),
 'epoch_starts': array([  0,  20,  39,  58,  78,  97, 116, 136, 155, 175, 194, 213, 233,
        252, 271, 291, 310, 330, 349, 368, 388, 407, 426, 446, 465, 485]),
 'normalized_errors': array([2.25481611e-07, 2.19217091e-07, 1.61692121e-07, 2.00011689e-07,
        1.89729824e-07, 2.13666591e-07, 1.89567973e-07, 1.69287402e-07,
        1.73439003e-07, 2.54446728e-07, 1.37637335e-07, 2.03208686e-07,
        1.82528374e-07, 1.52313657e-07, 1.31664943e-07, 1.31717232e-07,
        1.40488251e-07, 1.79658451e-07, 2.09202977e-07, 2.27380629e-07,
        1.79770641e-07, 1.44962979e-07, 2.15742776e-07, 1.45915585e-07,
        1.65332482e-07]),
 'absolute_errors': array([2.38418579e-13, 3.57627869e-13, 3.51923518e-13, 4.76837159e-13,
        4.17232512e-13, 5.06639481e-13, 8.34465026e-13, 4.17232514e-13,
        3.57627869e-13, 3.57627869e-13, 2.38418580e-13, 3